In [ ]:
import os
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

# Cấu hình đường dẫn gốc
ROOT_DIR = Path(".").resolve()
DATA_PATH = ROOT_DIR / "data_cleaned" / "rfm_model_features.csv"
FIGURES_DIR = ROOT_DIR / "reports" / "figures"
MODELS_DIR = ROOT_DIR / "models" / "outputs" / "final"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("-> Đã khởi tạo môi trường thành công!")

In [ ]:
df_scaled = pd.read_csv(DATA_PATH)
features = ['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled']
X = df_scaled[features].values
df_scaled.head()

In [ ]:
n_components_range = range(2, 11)
bic_scores, aic_scores = [], []

for k in n_components_range:
    gmm = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
    gmm.fit(X)
    bic_scores.append(gmm.bic(X))
    aic_scores.append(gmm.aic(X))

plt.figure(figsize=(10, 5))
plt.plot(n_components_range, bic_scores, label='BIC', marker='o')
plt.plot(n_components_range, aic_scores, label='AIC', marker='s')
plt.xlabel('Số lượng cụm (n_components)')
plt.ylabel('Score')
plt.title('Lựa chọn số cụm GMM tối ưu bằng BIC & AIC')
plt.legend()
plt.grid(True)
plt.savefig(FIGURES_DIR / "gmm_bic_aic.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_k = 4
gmm_best = GaussianMixture(n_components=best_k, random_state=42, covariance_type='full')
clusters = gmm_best.fit_predict(X)
probs = gmm_best.predict_proba(X)

print(f"Silhouette Score: {silhouette_score(X, clusters):.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X, clusters):.4f}")

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=clusters, palette='Set1', alpha=0.7)
plt.title(f'Phân cụm GMM (k={best_k}) qua PCA')
plt.savefig(FIGURES_DIR / "gmm_clusters_pca.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. Lưu file pickle cho Streamlit
joblib.dump(gmm_best, MODELS_DIR / "gmm_model.pkl")

# 2. Xuất dữ liệu kèm xác suất
df_result = df_scaled.copy()
df_result['GMM_Cluster'] = clusters
for i in range(best_k):
    df_result[f'Prob_Cluster_{i}'] = probs[:, i]

df_result.to_csv(MODELS_DIR / "gmm_results.csv", index=False)
print("-> Đã lưu mô hình và kết quả dữ liệu thành công!")